# TRX Tuple-Level Set Matching — Experiments

Backup experiment notebook for running on Google Colab (A100).

## Experiment plan

| # | Experiment | Branch | Config |
|---|---|---|---|
| 1 | Base (prototype TRX) | main | — |
| 2 | Bidirectional Hausdorff | stage1 | stage1_bidirectional.yaml |
| 3 | Attention-weighted Hausdorff | stage1 | stage1_attention_weighted.yaml |
| 4 | Stage 2 Option A (frame-level relation) | stage2 | stage2_option_a.yaml |
| 5 | Stage 2 Option B (tuple-level relation) | stage2 | stage2_option_b.yaml |

**Run cells top to bottom. Each experiment cell blocks until that experiment finishes.**

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('CUDA:', torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA        = '/content/drive/MyDrive/trx_data'
DRIVE_CHECKPOINTS = '/content/drive/MyDrive/trx_checkpoints'

assert os.path.exists(DRIVE_DATA + '/video_datasets/data/hmdb51_256q5.zip'), \
    'Data zip not found at ' + DRIVE_DATA + '/video_datasets/data/hmdb51_256q5.zip'
# splits are included in the repo — no need to put them on Drive

os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
print('Drive mounted and data verified.')

## 2. Copy data to local storage

Reading from Drive during training is slow (~1 MB/s). Copying to Colab local SSD
first gives ~10x faster I/O and prevents the GPU from waiting on data.

In [ ]:
import shutil, time

LOCAL_DATA = '/content/trx_data'
os.makedirs(LOCAL_DATA + '/video_datasets/data',   exist_ok=True)
os.makedirs(LOCAL_DATA + '/video_datasets/splits', exist_ok=True)

# Copy zip from Drive (~10-20 GB, takes 5-15 min)
dst_zip = LOCAL_DATA + '/video_datasets/data/hmdb51_256q5.zip'
if not os.path.exists(dst_zip):
    print('Copying dataset zip to local storage (this takes a few minutes)...')
    t0 = time.time()
    shutil.copy2(DRIVE_DATA + '/video_datasets/data/hmdb51_256q5.zip', dst_zip)
    print(f'Done in {(time.time()-t0)/60:.1f} min')
else:
    print('Zip already on local storage.')

# Copy splits from cloned repo (must run AFTER the clone cell)
dst_splits = LOCAL_DATA + '/video_datasets/splits/hmdb_ARN'
if not os.path.exists(dst_splits):
    shutil.copytree(REPO_DIR + '/splits/hmdb_ARN', dst_splits)
    print('Splits copied from repo.')
else:
    print('Splits already on local storage.')

print('Local data ready at', LOCAL_DATA)

## 3. Clone repo and install dependencies

In [ ]:
REPO_URL = 'https://github.com/38563541/TheRandomXeno.git'
REPO_DIR = '/content/TheRandomXeno'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

print('Repo ready.')

In [ ]:
!pip install pyyaml tensorboard --quiet

import tensorflow as tf
tf.compat.v1.disable_eager_execution()
print('TF version:', tf.__version__, '| Eager disabled:', not tf.compat.v1.executing_eagerly())

## 4. Helper: run one experiment

In [ ]:
import subprocess, sys, datetime

def run_experiment(name, branch, extra_args, split=3):
    checkpoint_dir = DRIVE_CHECKPOINTS + '/' + name + '_hmdb' + str(split)
    log_path       = DRIVE_CHECKPOINTS + '/' + name + '_hmdb' + str(split) + '.log'
    os.makedirs(checkpoint_dir, exist_ok=True)

    subprocess.run(['git', 'checkout', branch], cwd=REPO_DIR, check=True, capture_output=True)
    print('[' + name + '] Branch:', branch)

    base_args = [
        sys.executable, 'run.py',
        '--dataset', 'hmdb',
        '--split',   str(split),
'--test_iters', '25000', '50000', '75000', '100000',
        '--scratch', LOCAL_DATA,
        '-c', checkpoint_dir,
    ]
    cmd = base_args + extra_args

    print('[' + name + '] Starting at', datetime.datetime.now().strftime('%H:%M:%S'))
    print('[' + name + '] Checkpoint:', checkpoint_dir)
    print('[' + name + '] Log:', log_path)
    print('-' * 60)

    t0 = time.time()
    with open(log_path, 'w') as log_f:
        proc = subprocess.Popen(
            cmd, cwd=REPO_DIR,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in proc.stdout:
            print(line, end='')
            log_f.write(line)
            log_f.flush()
        proc.wait()

    elapsed = (time.time() - t0) / 3600
    print('\n[' + name + '] Finished in', round(elapsed, 2), 'h | Exit code:', proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError('Experiment ' + name + ' failed with exit code ' + str(proc.returncode))

print('Helper ready.')

---
## Experiment 1 — Base (original prototype TRX)
Branch: `main` | No config file

In [ ]:
run_experiment(
    name   = 'base_proto',
    branch = 'main',
    extra_args = [
        '--method',               'resnet18',
        '--trans_linear_out_dim', '1152',
        '--way',                  '5',
        '--shot',                 '1',
        '--query_per_class',      '5',
        '--tasks_per_batch',      '16',
        '--training_iterations',  '100020',
    ]
)

---
## Experiment 2 — Bidirectional Hausdorff
Branch: `stage1` | Config: `stage1_bidirectional.yaml`

In [ ]:
run_experiment(
    name       = 'abl_bidir',
    branch     = 'stage1',
    extra_args = ['--config', 'configs/stage1_bidirectional.yaml']
)

---
## Experiment 3 — Attention-weighted Hausdorff
Branch: `stage1` | Config: `stage1_attention_weighted.yaml`

In [ ]:
run_experiment(
    name       = 'abl_attn',
    branch     = 'stage1',
    extra_args = ['--config', 'configs/stage1_attention_weighted.yaml']
)

---
## Experiment 4 — Stage 2 Option A (relation at frame level)
Branch: `stage2` | Config: `stage2_option_a.yaml`

In [ ]:
run_experiment(
    name       = 'stage2_optA',
    branch     = 'stage2',
    extra_args = ['--config', 'configs/stage2_option_a.yaml']
)

---
## Experiment 5 — Stage 2 Option B (relation at tuple level)
Branch: `stage2` | Config: `stage2_option_b.yaml`

In [ ]:
run_experiment(
    name       = 'stage2_optB',
    branch     = 'stage2',
    extra_args = ['--config', 'configs/stage2_option_b.yaml']
)

---
## Results summary

In [ ]:
import csv, glob

csv_path = REPO_DIR + '/experiments/results/results.csv'
if os.path.exists(csv_path):
    shutil.copy2(csv_path, DRIVE_CHECKPOINTS + '/results.csv')
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    header = '{:<45} {:>8} {:>8} {:>8}'.format('Config', 'iter', 'acc', 'CI')
    print(header)
    print('-' * 75)
    for r in rows:
        cfg = r['config_file'].replace('configs/', '').replace('.yaml', '')
        print('{:<45} {:>8} {:>8} {:>8}'.format(cfg, r['iteration'], r['mean_accuracy'], r['confidence_interval']))
else:
    print('results.csv not found yet.')

print('\n=== Log summaries ===')
for log in sorted(glob.glob(DRIVE_CHECKPOINTS + '/*.log')):
    name = os.path.basename(log)
    lines = open(log).readlines()
    test_lines = [l for l in lines if 'accuracy' in l.lower() and 'test' in l.lower()]
    print('\n--- ' + name + ' ---')
    for l in test_lines[-4:]:
        print(l, end='')

---
## Resume a crashed experiment

If Colab disconnects mid-run, re-mount Drive and run this cell to resume.

In [ ]:
RESUME_NAME   = 'abl_bidir'
RESUME_BRANCH = 'stage1'
RESUME_CONFIG = 'configs/stage1_bidirectional.yaml'

run_experiment(
    name       = RESUME_NAME,
    branch     = RESUME_BRANCH,
    extra_args = ['--config', RESUME_CONFIG, '-r']
)